In [3]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

In [4]:
slam_file_tum = "f_dataset-loc000_monoic.tum"
gt_file = "/home/brg/Dev/mif/MagPIE/dataset/Ground_Truth_Evaluation/magpie2Dataset_loc000_mocap.csv"
slam_file_euroc = "evo_loc000.csv"

In [5]:
def tum_to_euroc(tum_file, euroc_file):

    data = np.loadtxt(tum_file)

    data[:, 0] *= 1e9
    df = pd.DataFrame(data, columns=["timestamp", "x", "y", "z", "qx", "qy", "qz", "qw"])
    return df
    # Save as CSV (comma-separated)
    # df.to_csv(euroc_file, index=False)

In [6]:
df = tum_to_euroc(slam_file_tum, slam_file_euroc)

In [7]:
def interpolate_gt(gt_file, slam_file, df_slam):

    gt_data = np.loadtxt(gt_file, delimiter = ',', skiprows=1)
    slam_data = np.loadtxt(slam_file, delimiter = ',', skiprows = 1)

    sxyz = slam_data[:, 1:4]
    gt_t, gt_xyz = gt_data[:, 0], gt_data[:, 1:4]  # Extract timestamps and positions
    # slam_t = slam_data[:, 0]  # Extract SLAM timestamps
    slam_t = df_slam["timestamp"]

    if np.max(gt_t) > 1e12:
        gt_t /= 1e9
    if np.max(slam_t) > 1e12:
        slam_t /= 1e9

    interp_x = interp1d(gt_t, gt_xyz[:, 0], kind='linear', fill_value="extrapolate")
    interp_y = interp1d(gt_t, gt_xyz[:, 1], kind='linear', fill_value="extrapolate")
    interp_z = interp1d(gt_t, gt_xyz[:, 2], kind='linear', fill_value="extrapolate")

# sxyz[:, 0], sxyz[:, 1], sxyz[:, 2]
    return interp_x(slam_t), interp_y(slam_t), interp_z(slam_t), df_slam["x"], df_slam["y"], df_slam["z"],

In [8]:
gtx, gty, gtz, sx, sy, sz = interpolate_gt(gt_file, slam_file_euroc, df)

gtxyz = np.column_stack([gtx, gty, gtz])
sxyz = np.column_stack([sx, sy, sz])
gtxyz

array([[-2.09510151,  2.07813196,  1.21771792],
       [-2.09169591,  2.0611222 ,  1.21762707],
       [-2.08634186,  2.03545197,  1.21788584],
       ...,
       [ 0.08151188,  0.19135147,  1.21768286],
       [ 0.08175229,  0.18678002,  1.21793458],
       [ 0.08195949,  0.18206649,  1.21768659]])

In [9]:
msate = np.mean(np.linalg.norm(sxyz - gtxyz, axis = 1)**2)

In [10]:
msate

0.753828942633759